# Face Mask Detection: Full CV Pipeline From Scratch

Notebook ini membangun pipeline computer vision lengkap untuk klasifikasi penggunaan masker wajah. Seluruh model deep learning dibuat manual menggunakan layer dasar `Conv2D` dan dilatih dari awal tanpa bobot eksternal.

## 1. Desain Eksperimen

Dataset yang digunakan adalah Face Mask Dataset dari Kaggle dengan dua kelas utama: `with_mask` dan `without_mask`. Split dibuat sekali di level file asli dengan rasio 70/15/15 untuk train, validation, dan test. Semua skenario memakai split yang sama agar tidak terjadi data leakage.

Pipeline akademik yang diuji:

- preprocessing: resize 128x128, normalisasi 0-1, label binary
- enhancement/restoration: CLAHE dan Gaussian denoise ringan
- face segmentation/cropping: Haar Cascade dengan fallback ke full image
- feature extraction klasik: HOG
- classification klasik: SVM
- classification deep learning: custom CNN Conv2D from scratch dengan opsi SE block
- evaluasi: accuracy, precision, recall, F1-score, classification report, confusion matrix, ROC curve, AUC, dan kurva training

Skenario wajib:

- HOG+SVM vs CNN
- enhancement vs tanpa enhancement
- crop vs full image
- augmentation vs tanpa augmentation
- SE block vs tanpa SE block

Catatan penting: notebook ini tidak memakai model siap pakai, metode pemindahan bobot, atau bobot eksternal. Arsitektur CNN dibuat manual dari layer `Conv2D`.

## 2. Setup

Konfigurasi dibuat eksplisit agar eksperimen mudah direproduksi dan mudah dijelaskan saat presentasi/laporan.

In [ ]:
import json
import os
import random
import shutil
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
MAX_EPOCHS = 50
EARLY_STOP_PATIENCE = 8
LR_PATIENCE = 3

RUN_OPTIONAL_CROP_SCENARIO = True
RUN_SE_ABLATION = True
RUN_LIGHT_TUNING = True

CLASS_NAMES = ['with_mask', 'without_mask']
CLASS_TO_LABEL = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

OUTPUT_DIR = Path('/kaggle/working')
TEMP_DIR = Path('/kaggle/temp')
DEFAULT_DATASET_DIR = Path('/kaggle/input/face-mask-dataset')
SCENARIO_ROOT = TEMP_DIR / 'face_mask_scenarios_final'
DATASET_HANDLE = 'omkargurav/face-mask-dataset'

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 3. Load Dataset Kaggle

Notebook mencari folder dataset yang berisi class `with_mask` dan `without_mask`. Jika dataset belum ter-mount di Kaggle, fallback download disediakan untuk menjaga notebook tetap reproducible.

In [ ]:
def print_tree(root: Path, max_depth: int = 3, limit: int = 120) -> None:
    print(f'Dataset tree preview: {root}')
    if not root.exists():
        print('Dataset path does not exist.')
        return

    root_depth = len(root.parts)
    shown = 0
    for path in sorted(root.rglob('*')):
        depth = len(path.parts) - root_depth
        if depth > max_depth:
            continue
        indent = '  ' * max(depth - 1, 0)
        suffix = '/' if path.is_dir() else ''
        print(f'{indent}{path.name}{suffix}')
        shown += 1
        if shown >= limit:
            print('... tree preview truncated ...')
            break


def find_image_directory(dataset_root: Path) -> Path:
    directories = [dataset_root]
    if dataset_root.exists():
        directories.extend([p for p in dataset_root.rglob('*') if p.is_dir()])

    for directory in directories:
        child_dirs = {p.name.lower(): p for p in directory.iterdir() if p.is_dir()}
        if all(name in child_dirs for name in CLASS_NAMES):
            return directory

    raise FileNotFoundError(f'Could not find class folders {CLASS_NAMES} under {dataset_root}')


def find_mounted_dataset() -> Path | None:
    if DEFAULT_DATASET_DIR.exists():
        return DEFAULT_DATASET_DIR

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        print('Available /kaggle/input entries:', [p.name for p in kaggle_input.iterdir()])
        for pattern in ['*face*mask*', '*mask*', '*face*']:
            matches = [p for p in kaggle_input.glob(pattern) if p.is_dir()]
            if matches:
                return matches[0]
    return None


def download_dataset_fallback() -> Path:
    download_dir = OUTPUT_DIR / 'downloaded_face_mask_dataset'
    download_dir.mkdir(parents=True, exist_ok=True)

    try:
        import kagglehub

        print('Dataset mount not found. Trying kagglehub fallback download...')
        downloaded_path = Path(kagglehub.dataset_download(DATASET_HANDLE))
        print('kagglehub downloaded dataset to:', downloaded_path)
        return downloaded_path
    except Exception as error:
        print('kagglehub fallback failed:', repr(error))

    print('Trying Kaggle CLI fallback download...')
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', DATASET_HANDLE, '-p', str(download_dir), '--unzip'],
        check=True,
    )
    return download_dir


dataset_root = find_mounted_dataset()
if dataset_root is None:
    dataset_root = download_dataset_fallback()

print_tree(dataset_root)
original_image_dir = find_image_directory(dataset_root)
print('Selected image directory:', original_image_dir)

## 4. Split 70/15/15 Tanpa Data Leakage

Split dilakukan sebelum preprocessing. Artinya satu file asli hanya masuk ke satu split, lalu turunan preprocessing untuk semua skenario dibuat dari daftar file tersebut.

In [ ]:
def collect_image_paths(image_dir: Path) -> dict[str, list[Path]]:
    paths_by_class = {}
    for class_name in CLASS_NAMES:
        class_dir = image_dir / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        paths = [p for p in sorted(class_dir.rglob('*')) if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
        if not paths:
            raise ValueError(f'No images found for class: {class_name}')
        paths_by_class[class_name] = paths
    return paths_by_class


def split_paths(paths_by_class: dict[str, list[Path]]) -> dict[str, list[tuple[Path, str]]]:
    rng = random.Random(SEED)
    splits = {'train': [], 'validation': [], 'test': []}

    for class_name, paths in paths_by_class.items():
        shuffled = paths.copy()
        rng.shuffle(shuffled)
        n_total = len(shuffled)
        n_train = int(n_total * TRAIN_RATIO)
        n_val = int(n_total * VAL_RATIO)

        splits['train'].extend((path, class_name) for path in shuffled[:n_train])
        splits['validation'].extend((path, class_name) for path in shuffled[n_train:n_train + n_val])
        splits['test'].extend((path, class_name) for path in shuffled[n_train + n_val:])

    for split_name in splits:
        rng.shuffle(splits[split_name])
    return splits


def count_split_labels(splits: dict[str, list[tuple[Path, str]]]) -> dict[str, dict[str, int]]:
    counts = {}
    for split_name, records in splits.items():
        counts[split_name] = {class_name: 0 for class_name in CLASS_NAMES}
        for _, class_name in records:
            counts[split_name][class_name] += 1
    return counts


paths_by_class = collect_image_paths(original_image_dir)
class_counts = {class_name: len(paths) for class_name, paths in paths_by_class.items()}
splits = split_paths(paths_by_class)
split_counts = count_split_labels(splits)
split_sizes = {split_name: len(records) for split_name, records in splits.items()}

all_split_paths = []
for records in splits.values():
    all_split_paths.extend(str(path.resolve()) for path, _ in records)
assert len(all_split_paths) == len(set(all_split_paths)), 'Data leakage detected: duplicated original path across splits.'

print('Class counts:', class_counts)
print('Split sizes:', split_sizes)
print('Split label counts:', split_counts)

## 5. Enhancement, Restoration, dan Face Crop

Enhancement terdiri dari CLAHE dan Gaussian denoise ringan. CLAHE meningkatkan kontras lokal pada channel luminance LAB, sedangkan Gaussian blur ringan mereduksi noise tanpa menghapus struktur masker secara agresif. Face crop memakai Haar Cascade; jika deteksi gagal, gambar full tetap dipakai.

In [ ]:
def apply_clahe_rgb(image_rgb: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_l = clahe.apply(l_channel)
    enhanced_lab = cv2.merge((enhanced_l, a_channel, b_channel))
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2RGB)


def apply_gaussian_denoise(image_rgb: np.ndarray) -> np.ndarray:
    return cv2.GaussianBlur(image_rgb, ksize=(3, 3), sigmaX=0.5)


def crop_face_if_detected(image_rgb: np.ndarray, detector: cv2.CascadeClassifier) -> tuple[np.ndarray, bool]:
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(35, 35))
    if len(faces) == 0:
        return image_rgb, False

    x, y, w, h = max(faces, key=lambda box: box[2] * box[3])
    pad = int(0.18 * max(w, h))
    x1 = max(0, x - pad)
    y1 = max(0, y - pad)
    x2 = min(image_rgb.shape[1], x + w + pad)
    y2 = min(image_rgb.shape[0], y + h + pad)
    return image_rgb[y1:y2, x1:x2], True


def preprocess_image(
    image_path: Path,
    use_enhancement: bool = False,
    use_face_crop: bool = False,
    detector: cv2.CascadeClassifier | None = None,
) -> tuple[np.ndarray | None, bool]:
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        return None, False

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    face_detected = False
    if use_face_crop and detector is not None:
        image_rgb, face_detected = crop_face_if_detected(image_rgb, detector)
    if use_enhancement:
        image_rgb = apply_clahe_rgb(image_rgb)
        image_rgb = apply_gaussian_denoise(image_rgb)

    resized = cv2.resize(image_rgb, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    return resized, face_detected


def build_preprocessed_scenario(
    scenario_name: str,
    use_enhancement: bool = False,
    use_face_crop: bool = False,
) -> dict[str, int]:
    scenario_dir = SCENARIO_ROOT / scenario_name
    if scenario_dir.exists():
        shutil.rmtree(scenario_dir)

    cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
    detector = cv2.CascadeClassifier(str(cascade_path)) if use_face_crop else None
    stats = {'processed': 0, 'read_failed': 0, 'faces_detected': 0}

    for split_name, records in splits.items():
        for idx, (image_path, class_name) in enumerate(records):
            target_class_dir = scenario_dir / split_name / class_name
            target_class_dir.mkdir(parents=True, exist_ok=True)

            image_rgb, face_detected = preprocess_image(
                image_path,
                use_enhancement=use_enhancement,
                use_face_crop=use_face_crop,
                detector=detector,
            )
            if image_rgb is None:
                stats['read_failed'] += 1
                continue

            output_path = target_class_dir / f'{idx:06d}_{image_path.stem}.jpg'
            cv2.imwrite(str(output_path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
            stats['processed'] += 1
            stats['faces_detected'] += int(face_detected)

    return stats


SCENARIO_CONFIGS = {
    'full_plain': {'use_enhancement': False, 'use_face_crop': False},
    'full_enhanced': {'use_enhancement': True, 'use_face_crop': False},
    'crop_enhanced': {'use_enhancement': True, 'use_face_crop': True},
}

preprocessing_stats = {}
for scenario_name, config in SCENARIO_CONFIGS.items():
    preprocessing_stats[scenario_name] = build_preprocessed_scenario(scenario_name, **config)

print('Preprocessing stats:', preprocessing_stats)
print_tree(SCENARIO_ROOT, max_depth=3, limit=90)

## 6. Visualisasi Dataset dan Preprocessing

Visualisasi ini membantu menjelaskan karakteristik dataset, masalah pencahayaan/background, dan efek enhancement/cropping secara kualitatif.

In [ ]:
def read_rgb(path: Path) -> np.ndarray:
    image_bgr = cv2.imread(str(path))
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


preview_records = []
for class_name in CLASS_NAMES:
    preview_records.extend((path, class_name) for path in paths_by_class[class_name][:2])

cascade_path = Path(cv2.data.haarcascades) / 'haarcascade_frontalface_default.xml'
preview_detector = cv2.CascadeClassifier(str(cascade_path))

plt.figure(figsize=(12, 10))
for row, (image_path, class_name) in enumerate(preview_records[:4]):
    original = read_rgb(image_path)
    plain, _ = preprocess_image(image_path, use_enhancement=False, use_face_crop=False)
    enhanced, _ = preprocess_image(image_path, use_enhancement=True, use_face_crop=False)
    crop_enhanced, detected = preprocess_image(
        image_path,
        use_enhancement=True,
        use_face_crop=True,
        detector=preview_detector,
    )
    images = [original, plain, enhanced, crop_enhanced]
    titles = [f'Original: {class_name}', 'Resize 128x128', 'CLAHE + Gaussian', f'Crop + enhance ({detected})']

    for col, (image, title) in enumerate(zip(images, titles)):
        plt.subplot(4, 4, row * 4 + col + 1)
        plt.imshow(image)
        plt.title(title)
        plt.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'preprocessing_examples.png', dpi=180)
plt.show()

plt.figure(figsize=(7, 4))
bars = plt.bar(class_counts.keys(), class_counts.values(), color=['#2f80ed', '#27ae60'])
plt.title('Jumlah Gambar per Class')
plt.xlabel('Class')
plt.ylabel('Jumlah gambar')
plt.bar_label(bars)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=160)
plt.show()

## 7. Dataset Loader, Normalisasi, dan Augmentation

CNN menerima input float 0-1 dan label binary. Augmentation realistis hanya diterapkan pada data training: horizontal flip, rotation kecil, zoom ringan, contrast ringan, dan brightness ringan.

In [ ]:
geometry_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal', seed=SEED),
        tf.keras.layers.RandomRotation(0.07, seed=SEED),
        tf.keras.layers.RandomZoom(0.10, seed=SEED),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name='realistic_geometry_augmentation',
)


def load_tf_split(scenario_name: str, split_name: str, shuffle: bool = False, batch_size: int = BATCH_SIZE) -> tf.data.Dataset:
    return tf.keras.utils.image_dataset_from_directory(
        SCENARIO_ROOT / scenario_name / split_name,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        batch_size=batch_size,
        image_size=IMAGE_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )


def augment_images(images: tf.Tensor, labels: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    images = geometry_augmentation(images, training=True)
    images = tf.image.random_brightness(images, max_delta=18.0, seed=SEED)
    images = tf.clip_by_value(images, 0.0, 255.0)
    return images, labels


def normalize_images(images: tf.Tensor, labels: tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    return tf.cast(images, tf.float32) / 255.0, tf.cast(labels, tf.float32)


def prepare_cnn_dataset(dataset: tf.data.Dataset, augment: bool = False) -> tf.data.Dataset:
    if augment:
        dataset = dataset.map(augment_images, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.map(normalize_images, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.prefetch(tf.data.AUTOTUNE)


def load_images_for_classical(scenario_name: str, split_name: str) -> tuple[np.ndarray, np.ndarray]:
    images = []
    labels = []
    for class_name in CLASS_NAMES:
        class_dir = SCENARIO_ROOT / scenario_name / split_name / class_name
        for image_path in sorted(class_dir.glob('*.jpg')):
            images.append(read_rgb(image_path))
            labels.append(CLASS_TO_LABEL[class_name])
    return np.array(images, dtype=np.uint8), np.array(labels, dtype=np.int64)

## 8. Feature Extraction Klasik: HOG + SVM

HOG dipilih karena merepresentasikan orientasi tepi dan bentuk lokal. SVM digunakan sebagai classifier klasik yang kuat untuk fitur manual. Baseline ini penting agar performa CNN dapat dibandingkan dengan pendekatan computer vision klasik.

In [ ]:
def extract_hog_features(images: np.ndarray) -> np.ndarray:
    hog = cv2.HOGDescriptor(
        (128, 128),
        (16, 16),
        (8, 8),
        (8, 8),
        9,
    )
    features = []
    for image in images:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        descriptor = hog.compute(gray).flatten()
        features.append(descriptor)
    return np.array(features, dtype=np.float32)


def plot_confusion_matrix(matrix: np.ndarray, scenario_id: str) -> None:
    fig, ax = plt.subplots(figsize=(5, 5))
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=CLASS_NAMES)
    display.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    plt.title(f'Confusion Matrix - {scenario_id}')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confusion_matrix_{scenario_id}.png', dpi=180)
    plt.show()


def plot_roc_curve(y_true: np.ndarray, y_score: np.ndarray, scenario_id: str) -> float:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5.5, 5))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {scenario_id}')
    plt.grid(alpha=0.3)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'roc_curve_{scenario_id}.png', dpi=180)
    plt.show()
    return float(roc_auc)


def evaluate_predictions(
    scenario_id: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
    model_type: str,
    config: dict | None = None,
    history: dict | None = None,
    test_loss: float | None = None,
) -> dict:
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    roc_auc = plot_roc_curve(y_true, y_score, scenario_id)

    result = {
        'scenario': scenario_id,
        'model_type': model_type,
        'config': config or {},
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': roc_auc,
        'classification_report': report_dict,
        'confusion_matrix': matrix.tolist(),
    }
    if history is not None:
        result['history'] = {key: [float(v) for v in values] for key, values in history.items()}
    if test_loss is not None:
        result['test_loss'] = float(test_loss)

    print(f'\n=== {scenario_id} ===')
    print(f"Accuracy : {result['accuracy']:.4f}")
    print(f"Precision: {result['precision']:.4f}")
    print(f"Recall   : {result['recall']:.4f}")
    print(f"F1-score : {result['f1_score']:.4f}")
    print(f"AUC      : {result['auc']:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    plot_confusion_matrix(matrix, scenario_id)
    return result


scenario_results = []

for scenario_id, scenario_name in [
    ('A_hog_svm_plain_full', 'full_plain'),
    ('B_hog_svm_enhanced_full', 'full_enhanced'),
]:
    X_train_img, y_train = load_images_for_classical(scenario_name, 'train')
    X_val_img, y_val = load_images_for_classical(scenario_name, 'validation')
    X_test_img, y_test = load_images_for_classical(scenario_name, 'test')

    X_train = extract_hog_features(X_train_img)
    X_val = extract_hog_features(X_val_img)
    X_test = extract_hog_features(X_test_img)

    svm_model = make_pipeline(
        StandardScaler(),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=8000, random_state=SEED),
    )
    svm_model.fit(X_train, y_train)
    val_pred = svm_model.predict(X_val)
    print(f'Validation accuracy {scenario_id}: {accuracy_score(y_val, val_pred):.4f}')

    y_score = svm_model.decision_function(X_test)
    y_pred = (y_score >= 0).astype(np.int64)
    scenario_results.append(
        evaluate_predictions(
            scenario_id,
            y_test,
            y_pred,
            y_score,
            model_type='HOG + LinearSVC',
            config={'feature': 'HOG', 'classifier': 'LinearSVC', 'enhancement': scenario_name == 'full_enhanced'},
        )
    )

## 9. Custom CNN From Scratch dengan Optional SE Block

Arsitektur CNN dibuat manual dari `Conv2D`, BatchNormalization, ReLU, MaxPooling, Dropout, GlobalAveragePooling, dan Dense. SE block ringan ditambahkan setelah Block 3 dan Block 4 untuk memberi attention channel tanpa memakai bobot eksternal.

In [ ]:
def make_optimizer(name: str, learning_rate: float):
    name = name.lower()
    if name == 'adamw' and hasattr(tf.keras.optimizers, 'AdamW'):
        return tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)


def conv_bn_relu(x: tf.Tensor, filters: int) -> tf.Tensor:
    x = tf.keras.layers.Conv2D(filters, kernel_size=3, padding='same', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return tf.keras.layers.ReLU()(x)


def squeeze_excite_block(x: tf.Tensor, ratio: int = 8) -> tf.Tensor:
    channels = int(x.shape[-1])
    se = tf.keras.layers.GlobalAveragePooling2D()(x)
    se = tf.keras.layers.Dense(max(channels // ratio, 8), activation='relu')(se)
    se = tf.keras.layers.Dense(channels, activation='sigmoid')(se)
    se = tf.keras.layers.Reshape((1, 1, channels))(se)
    return tf.keras.layers.Multiply()([x, se])


def build_custom_cnn(config: dict) -> tf.keras.Model:
    filters = config.get('filters', [32, 64, 128, 256])
    dropout_scale = float(config.get('dropout_scale', 1.0))
    use_se = bool(config.get('use_se', True))
    dense_units = int(config.get('dense_units', 128))
    optimizer_name = config.get('optimizer', 'adamw')
    learning_rate = float(config.get('learning_rate', 1e-3))

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = inputs
    block_dropouts = [0.15, 0.20, 0.25, 0.30]

    for block_idx, filter_count in enumerate(filters):
        x = conv_bn_relu(x, filter_count)
        x = conv_bn_relu(x, filter_count)
        if use_se and block_idx in [2, 3]:
            x = squeeze_excite_block(x)
        x = tf.keras.layers.MaxPooling2D()(x)
        x = tf.keras.layers.Dropout(block_dropouts[block_idx] * dropout_scale)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(dense_units, activation='relu', kernel_initializer='he_normal')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.40 * dropout_scale)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs, name='custom_cnn_from_scratch')
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
    )
    return model


base_cnn_config = {
    'filters': [32, 64, 128, 256],
    'dropout_scale': 1.0,
    'use_se': True,
    'dense_units': 128,
    'optimizer': 'adamw',
    'learning_rate': 1e-3,
    'batch_size': 32,
}

build_custom_cnn(base_cnn_config).summary()

## 10. Training CNN, ROC/AUC, dan Model Checkpoint

Model disimpan berdasarkan validation loss. ReduceLROnPlateau membantu optimasi ketika validation loss stagnan, sedangkan EarlyStopping mencegah overfitting berlebihan.

In [ ]:
def plot_training_history(history: dict, scenario_id: str) -> None:
    plt.figure(figsize=(11, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.get('accuracy', []), marker='o', label='train')
    plt.plot(history.get('val_accuracy', []), marker='o', label='validation')
    plt.title(f'Accuracy - {scenario_id}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.get('loss', []), marker='o', label='train')
    plt.plot(history.get('val_loss', []), marker='o', label='validation')
    plt.title(f'Loss - {scenario_id}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'training_history_{scenario_id}.png', dpi=180)
    plt.show()


def train_and_evaluate_cnn(scenario_id: str, scenario_name: str, config: dict, augment_train: bool = False) -> tuple[dict, tf.keras.Model]:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    batch_size = int(config.get('batch_size', BATCH_SIZE))
    train_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'train', shuffle=True, batch_size=batch_size), augment=augment_train)
    val_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'validation', shuffle=False, batch_size=batch_size))
    test_ds = prepare_cnn_dataset(load_tf_split(scenario_name, 'test', shuffle=False, batch_size=batch_size))

    model = build_custom_cnn(config)
    checkpoint_path = OUTPUT_DIR / f'best_{scenario_id}.keras'
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
            mode='min',
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.4,
            patience=LR_PATIENCE,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True,
        ),
    ]

    print(f'\nTraining {scenario_id}')
    print('Config:', config, 'augment_train:', augment_train)
    history_obj = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    history = history_obj.history
    plot_training_history(history, scenario_id)

    test_metrics = model.evaluate(test_ds, verbose=0)
    metric_names = model.metrics_names
    test_metric_dict = {name: float(value) for name, value in zip(metric_names, test_metrics)}

    y_true = []
    y_score = []
    for images, labels in test_ds:
        scores = model.predict(images, verbose=0).reshape(-1)
        y_true.extend(labels.numpy().reshape(-1).astype(np.int64).tolist())
        y_score.extend(scores.tolist())

    y_true = np.array(y_true, dtype=np.int64)
    y_score = np.array(y_score, dtype=np.float32)
    y_pred = (y_score >= 0.5).astype(np.int64)

    result = evaluate_predictions(
        scenario_id,
        y_true,
        y_pred,
        y_score,
        model_type='Custom CNN Conv2D from scratch',
        config={**config, 'augmentation': bool(augment_train), 'scenario_dataset': scenario_name},
        history=history,
        test_loss=test_metric_dict.get('loss'),
    )
    result['keras_test_metrics'] = test_metric_dict
    result['checkpoint_path'] = str(checkpoint_path)
    result['epochs_trained'] = len(history.get('loss', []))
    result['best_val_loss'] = float(min(history.get('val_loss', [np.inf])))
    result['best_val_accuracy'] = float(max(history.get('val_accuracy', [0.0])))
    return result, model

## 11. Eksperimen Wajib dan Tuning Ringan

Eksperimen dibuat cukup lengkap tetapi tetap realistis untuk Kaggle. Tuning ringan mencakup learning rate, dropout, optimizer, augmentation, dan jumlah filter. Skenario SE vs tanpa SE dibuat sebagai ablation.

In [ ]:
cnn_experiments = [
    {
        'scenario_id': 'C_cnn_plain_full_se_no_aug',
        'scenario_name': 'full_plain',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'CNN tanpa enhancement',
    },
    {
        'scenario_id': 'D_cnn_enhanced_full_se_no_aug',
        'scenario_name': 'full_enhanced',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh enhancement',
    },
    {
        'scenario_id': 'F_cnn_enhanced_full_se_aug',
        'scenario_name': 'full_enhanced',
        'augment': True,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh augmentation',
    },
]

if RUN_OPTIONAL_CROP_SCENARIO:
    cnn_experiments.append({
        'scenario_id': 'E_cnn_enhanced_crop_se_no_aug',
        'scenario_name': 'crop_enhanced',
        'augment': False,
        'config': {**base_cnn_config, 'use_se': True},
        'purpose': 'Pengaruh face crop',
    })

if RUN_SE_ABLATION:
    cnn_experiments.append({
        'scenario_id': 'G_cnn_enhanced_full_no_se_aug',
        'scenario_name': 'full_enhanced',
        'augment': True,
        'config': {**base_cnn_config, 'use_se': False},
        'purpose': 'Ablation SE block',
    })

if RUN_LIGHT_TUNING:
    cnn_experiments.extend([
        {
            'scenario_id': 'T1_cnn_enhanced_full_se_aug_lr5e4',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'learning_rate': 5e-4, 'dropout_scale': 1.0, 'optimizer': 'adamw'},
            'purpose': 'Tuning learning rate',
        },
        {
            'scenario_id': 'T2_cnn_enhanced_full_se_aug_wider',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'filters': [48, 96, 160, 256], 'learning_rate': 7e-4, 'dropout_scale': 1.1, 'optimizer': 'adamw'},
            'purpose': 'Tuning jumlah filter dan dropout',
        },
        {
            'scenario_id': 'T3_cnn_enhanced_full_se_aug_adam',
            'scenario_name': 'full_enhanced',
            'augment': True,
            'config': {**base_cnn_config, 'optimizer': 'adam', 'learning_rate': 1e-3, 'dropout_scale': 0.9, 'batch_size': 64},
            'purpose': 'Tuning optimizer dan batch size',
        },
    ])

trained_models = {}
for experiment in cnn_experiments:
    result, model = train_and_evaluate_cnn(
        experiment['scenario_id'],
        experiment['scenario_name'],
        experiment['config'],
        augment_train=experiment['augment'],
    )
    result['purpose'] = experiment['purpose']
    scenario_results.append(result)
    trained_models[experiment['scenario_id']] = model

## 12. Perbandingan Semua Skenario

Bagian ini merangkum performa HOG+SVM dan CNN, pengaruh enhancement, crop, augmentation, SE block, serta hasil tuning ringan.

In [ ]:
def plot_scenario_comparison(results: list[dict]) -> None:
    names = [result['scenario'] for result in results]
    accuracy = [result['accuracy'] for result in results]
    precision = [result['precision'] for result in results]
    recall = [result['recall'] for result in results]
    f1 = [result['f1_score'] for result in results]
    auc_scores = [result['auc'] for result in results]

    x = np.arange(len(names))
    width = 0.16
    plt.figure(figsize=(16, 6))
    plt.bar(x - 2 * width, accuracy, width, label='accuracy')
    plt.bar(x - width, precision, width, label='precision')
    plt.bar(x, recall, width, label='recall')
    plt.bar(x + width, f1, width, label='f1-score')
    plt.bar(x + 2 * width, auc_scores, width, label='AUC')
    plt.xticks(x, names, rotation=30, ha='right')
    plt.ylim(0, 1.05)
    plt.ylabel('Score')
    plt.title('Perbandingan Semua Skenario')
    plt.grid(axis='y', alpha=0.3)
    plt.legend(ncol=5)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'scenario_comparison.png', dpi=180)
    plt.show()


def summarize_results(results: list[dict]) -> list[dict]:
    rows = []
    for result in results:
        rows.append({
            'scenario': result['scenario'],
            'model_type': result['model_type'],
            'accuracy': round(result['accuracy'], 4),
            'precision': round(result['precision'], 4),
            'recall': round(result['recall'], 4),
            'f1_score': round(result['f1_score'], 4),
            'auc': round(result['auc'], 4),
            'epochs': result.get('epochs_trained'),
            'best_val_loss': None if result.get('best_val_loss') is None else round(result['best_val_loss'], 4),
            'best_val_accuracy': None if result.get('best_val_accuracy') is None else round(result['best_val_accuracy'], 4),
            'purpose': result.get('purpose', ''),
            'config': result.get('config', {}),
        })
    return rows


summary_rows = summarize_results(scenario_results)
print(json.dumps(summary_rows, indent=2))
plot_scenario_comparison(scenario_results)

best_test_result = max(scenario_results, key=lambda item: (item['f1_score'], item['auc'], item['accuracy']))
cnn_results = [result for result in scenario_results if result['model_type'] == 'Custom CNN Conv2D from scratch']
best_cnn_result = min(cnn_results, key=lambda item: item.get('best_val_loss', np.inf))
print('Best test scenario for reporting:', best_test_result['scenario'])
print('Final selected CNN scenario by validation loss:', best_cnn_result['scenario'])
print('Final selected CNN config:', json.dumps(best_cnn_result['config'], indent=2))

## 13. Analisis Akademik

Dataset dipilih karena sesuai langsung dengan masalah klasifikasi penggunaan masker dan memiliki dua kelas yang jelas. Karakteristik dataset meliputi variasi pose, jarak wajah, background, pencahayaan, kualitas gambar, dan kemungkinan oklusi selain masker. Variasi ini membuat pipeline computer vision perlu diuji dari preprocessing hingga evaluasi akhir.

Masalah pencahayaan dan noise ditangani dengan enhancement/restoration ringan. CLAHE meningkatkan kontras lokal pada channel luminance sehingga detail area wajah dan masker dapat lebih terlihat, sedangkan Gaussian denoise ringan mengurangi noise tanpa merusak bentuk masker. Face crop bertujuan mengurangi pengaruh background, tetapi tetap memiliki risiko jika Haar Cascade gagal mendeteksi wajah pada pose ekstrem atau citra blur.

HOG digunakan sebagai baseline klasik karena menangkap distribusi orientasi tepi dan bentuk lokal. Baseline ini penting untuk menunjukkan apakah fitur manual masih kompetitif dibanding fitur yang dipelajari otomatis. CNN from scratch digunakan karena requirement melarang bobot eksternal; seluruh representasi dipelajari langsung dari dataset praktikum.

SE block digunakan sebagai attention channel ringan. Tujuannya adalah membantu model memberi bobot lebih besar pada channel fitur yang relevan tanpa menambah model eksternal. Ablation tanpa SE diperlukan untuk melihat apakah attention benar-benar membantu atau justru menambah kompleksitas.

Augmentation realistis membantu generalisasi terhadap variasi kecil seperti arah wajah, zoom, kontras, dan brightness. Augmentation ekstrem dihindari karena dapat mengubah bentuk masker atau membuat citra tidak realistis. Overfitting dianalisis dari jarak antara training dan validation accuracy/loss. Jika training accuracy tinggi tetapi validation loss naik, model mulai menghafal data. ReduceLROnPlateau, Dropout, BatchNormalization, dan EarlyStopping digunakan untuk menekan risiko tersebut.

Kekuatan sistem ini adalah pipeline lengkap, skenario jelas, dan evaluasi menyeluruh. Kelemahannya adalah Haar Cascade tidak selalu robust, tuning masih ringan, dan performa sangat bergantung pada kualitas/variasi dataset. Pengembangan lanjutan dapat mencakup cross-validation, threshold tuning berbasis validation set, analisis error per kondisi citra, balancing tambahan jika class tidak seimbang, dan face detector yang lebih stabil selama tetap sesuai aturan praktikum.

## 14. Simpan Artefak Final

Model CNN final dipilih berdasarkan validation loss terbaik dari skenario CNN yang dijalankan. Test set tetap dipakai untuk evaluasi akhir dan pelaporan, bukan untuk memilih model.

In [ ]:
best_model = trained_models[best_cnn_result['scenario']]
final_keras_path = OUTPUT_DIR / 'face_mask_custom_cnn_from_scratch_best.keras'
metrics_path = OUTPUT_DIR / 'metrics.json'
best_model.save(final_keras_path)

metrics = {
    'seed': SEED,
    'tensorflow_version': tf.__version__,
    'dataset_handle': DATASET_HANDLE,
    'dataset_root': str(dataset_root),
    'original_image_directory': str(original_image_dir),
    'class_names': CLASS_NAMES,
    'class_counts': class_counts,
    'split_ratio': {'train': TRAIN_RATIO, 'validation': VAL_RATIO, 'test': TEST_RATIO},
    'split_sizes': split_sizes,
    'split_label_counts': split_counts,
    'image_size': list(IMAGE_SIZE),
    'batch_size_default': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'preprocessing_stats': preprocessing_stats,
    'deep_learning_model': 'Custom CNN Conv2D from scratch',
    'best_test_scenario_for_reporting': best_test_result['scenario'],
    'best_cnn_scenario': best_cnn_result['scenario'],
    'best_cnn_config': best_cnn_result['config'],
    'summary_rows': summary_rows,
    'scenario_results': scenario_results,
    'outputs': {
        'best_keras_model': str(final_keras_path),
        'metrics': str(metrics_path),
        'class_distribution': str(OUTPUT_DIR / 'class_distribution.png'),
        'preprocessing_examples': str(OUTPUT_DIR / 'preprocessing_examples.png'),
        'scenario_comparison': str(OUTPUT_DIR / 'scenario_comparison.png'),
    },
}
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print('Saved outputs:')
for path in [
    final_keras_path,
    metrics_path,
    OUTPUT_DIR / 'class_distribution.png',
    OUTPUT_DIR / 'preprocessing_examples.png',
    OUTPUT_DIR / 'scenario_comparison.png',
]:
    if path.exists():
        print('-', path)